# enriquecimento de egressos com crossref

este notebook realiza o enriquecimento dos currículos dos egressos com metadados de publicações obtidos pela api pública da crossref.

o notebook utiliza como entrada os currículos lattes já baixados em formato xml e armazenados na pasta do projeto.

o orcid é extraído diretamente de cada currículo lattes e utilizado como identificador para consulta à crossref.

fluxo do módulo:

**xml lattes → orcid → crossref → publicações → indicadores → arquivo excel estruturado**

a consulta é feita pelo orcid sempre que disponível, reduzindo a dependência de buscas apenas pelo nome.

a estrutura das tabelas de saída segue o mesmo padrão utilizado no módulo openalex, facilitando a integração posterior entre as duas fontes.

---



## 1. instalar bibliotecas

nesta etapa são instaladas as bibliotecas utilizadas pelo notebook para leitura dos arquivos xml, manipulação dos dados, comunicação com a api da crossref e geração do arquivo excel.

In [ ]:
!pip -q install pandas openpyxl requests tqdm lxml

## 2. importar bibliotecas e conectar ao google drive

nesta etapa são importadas as bibliotecas utilizadas no processamento e realizada a conexão com o google drive, onde estão armazenados os currículos lattes e os arquivos do projeto.

In [ ]:
import os
import re
import time
import xml.etree.ElementTree as ET
from datetime import datetime

import pandas as pd
import requests

from tqdm.auto import tqdm
from google.colab import drive


drive.mount('/content/drive')

Mounted at /content/drive


## 3. configurar os diretórios

nesta etapa são definidos os diretórios utilizados pelo módulo.

os currículos lattes serão utilizados como arquivos de entrada.

os resultados produzidos pelo módulo crossref serão armazenados na pasta **estratégia 1**, dentro do diretório de enriquecimento de dados - crossref.

a organização dos diretórios mantém os arquivos de entrada separados dos resultados gerados pelo processamento.

In [ ]:
# pasta principal do projeto

DIRETORIO_PROJETO = (
    '/content/drive/MyDrive/RAGI 2026 2'
)

# pasta onde os resultados do crossref serão salvos

DIRETORIO_SAIDA = os.path.join(
    DIRETORIO_PROJETO,
    'Enriquecimento de dados - CROSSREF',
    'Estratégia 1'
)

# cria a pasta caso ela ainda não exista

os.makedirs(
    DIRETORIO_SAIDA,
    exist_ok=True
)

# arquivo excel final

ARQUIVO_SAIDA = os.path.join(
    DIRETORIO_SAIDA,
    'publicacoes_crossref_lattes.xlsx'
)

print('diretório de saída:')
print(DIRETORIO_SAIDA)

print()

print('arquivo final:')
print(ARQUIVO_SAIDA)

diretório de saída:
/content/drive/MyDrive/RAGI 2026 2/Enriquecimento de dados - CROSSREF/Estratégia 1

arquivo final:
/content/drive/MyDrive/RAGI 2026 2/Enriquecimento de dados - CROSSREF/Estratégia 1/publicacoes_crossref_lattes.xlsx


## 4. localizar os currículos lattes

nesta etapa o notebook percorre o diretório principal do projeto e suas subpastas para localizar os arquivos de currículo lattes armazenados em formato xml.

os arquivos encontrados serão utilizados como entrada para as próximas etapas do processamento.

In [ ]:
# procura todos os arquivos xml dentro do projeto

arquivos_xml = []

for raiz, diretorios, arquivos in os.walk(
    DIRETORIO_PROJETO
):
    for arquivo in arquivos:
        if arquivo.lower().endswith('.xml'):
            caminho_completo = os.path.join(
                raiz,
                arquivo
            )
            arquivos_xml.append(
                caminho_completo
            )

arquivos_xml = sorted(
    arquivos_xml
)

print(
    f'total de arquivos xml encontrados: '
    f'{len(arquivos_xml)}'
)
print()

for caminho in arquivos_xml[:20]:
    print(caminho)

total de arquivos xml encontrados: 271

/content/drive/MyDrive/RAGI 2026 2/Extrator de Currículo - LATTES/Currículos dos Egressos - LATTES/Abdul_Pedro_Manuel_Muchingeca_9666992319712718.xml
/content/drive/MyDrive/RAGI 2026 2/Extrator de Currículo - LATTES/Currículos dos Egressos - LATTES/Abner_Antônio_Ferreira_0596211008110321.xml
/content/drive/MyDrive/RAGI 2026 2/Extrator de Currículo - LATTES/Currículos dos Egressos - LATTES/Adriana_Teixeira_de_Moraes_3100726985926660.xml
/content/drive/MyDrive/RAGI 2026 2/Extrator de Currículo - LATTES/Currículos dos Egressos - LATTES/Adrielly_Campos_e_Almeida_4039893861746107.xml
/content/drive/MyDrive/RAGI 2026 2/Extrator de Currículo - LATTES/Currículos dos Egressos - LATTES/Aldenor_da_Silva_Pimentel_5770288768881926.xml
/content/drive/MyDrive/RAGI 2026 2/Extrator de Currículo - LATTES/Currículos dos Egressos - LATTES/Alessandra_Rodrigues_Oliveira_9237912840073975.xml
/content/drive/MyDrive/RAGI 2026 2/Extrator de Currículo - LATTE

## 5. identificar o egresso

os arquivos xml dos currículos lattes seguem uma estrutura de identificação no nome do arquivo.

nesta etapa o notebook utiliza o nome do arquivo para obter o nome do egresso e, quando disponível, o identificador do currículo lattes.

essas informações serão utilizadas para identificar o pesquisador nas tabelas geradas posteriormente.

In [ ]:
def extrair_orcid_xml(caminho_xml):
    """
    procura um identificador orcid dentro do currículo lattes.
    """
    try:
        tree = ET.parse(caminho_xml)
        root = tree.getroot()
        # procurar atributos contendo orcid
        for elemento in root.iter():
            for chave, valor in elemento.attrib.items():
                if 'orcid' in chave.lower():
                    orcid = normalizar_orcid(valor)
                    if orcid:
                        return orcid

        # procurar valores/textos contendo orcid
        for elemento in root.iter():
            if elemento.text:
                texto = elemento.text.strip()
                if 'orcid' in texto.lower():
                    encontrado = re.search(
                        r'\d{4}-\d{4}-\d{4}-[\dXx]{4}',
                        texto
                    )
                    if encontrado:
                        return normalizar_orcid(
                            encontrado.group(0)
                        )


        # busca geral no conteúdo textual do xml
        texto_completo = ET.tostring(
            root,
            encoding='unicode'
        )

        encontrado = re.search(
            r'\d{4}-\d{4}-\d{4}-[\dXx]{4}',
            texto_completo
        )

        if encontrado:
            return normalizar_orcid(
                encontrado.group(0)
            )

        return ''

    except Exception as erro:
        print(
            f'erro ao ler xml: {os.path.basename(caminho_xml)}'
        )
        print(erro)
        return ''

## 6. normalizar o orcid

nesta etapa é criada uma função para padronizar o identificador orcid encontrado nos currículos lattes.

o identificador será armazenado no formato padrão:

`0000-0000-0000-0000`

a normalização permite tratar diferentes formas de apresentação do mesmo orcid antes da consulta à crossref.

In [ ]:
def normalizar_orcid(valor):
    """
    normaliza um identificador orcid para o formato padrão.
    """
    if valor is None:
        return ''
    valor = str(valor).strip()

    if not valor:
        return ''
    valor = valor.replace(
        'https://orcid.org/',
        ''
    )
    valor = valor.replace(
        'http://orcid.org/',
        ''
    )
    valor = valor.strip().strip('/')

    if re.fullmatch(
        r'\d{4}-\d{4}-\d{4}-[\dXx]{4}',
        valor
    ):
        return valor.upper()

    return ''

## 7. extrair o orcid do currículo lattes

nesta etapa o conteúdo de cada arquivo xml é analisado para localizar o identificador orcid.

o orcid será extraído diretamente do currículo lattes, eliminando a necessidade de utilizar uma planilha intermediária apenas para armazenar esse identificador.

quando um orcid válido for encontrado, ele será retornado para utilização nas consultas à crossref.

In [ ]:
def extrair_identificacao_arquivo(nome_arquivo):

    nome_sem_extensao = os.path.splitext(
        nome_arquivo
    )[0]

    # procura uma sequência numérica no final do nome
    encontrado = re.search(
        r'_(\d{10,20})$',
        nome_sem_extensao
    )

    if encontrado:
        id_lattes = encontrado.group(1)
        nome = nome_sem_extensao[
            :encontrado.start()
        ]

    else:
        id_lattes = ''
        nome = nome_sem_extensao

    # converter underscores em espaços
    nome = nome.replace(
        '_',
        ' '
    )

    return nome.strip(), id_lattes

## 8. testar a leitura de um currículo

antes de processar todos os currículos, será realizado um teste utilizando um único arquivo xml.

o objetivo é verificar se o notebook consegue identificar corretamente:

- o nome do egresso;
- o id lattes;
- o orcid.

somente após essa validação será realizado o processamento completo dos currículos.

In [ ]:
arquivo_teste = arquivos_xml[0]

# Use diretamente a variável arquivo_teste (já é o caminho completo)
caminho_teste = arquivo_teste

nome_teste, id_lattes_teste = extrair_identificacao_arquivo(
    os.path.basename(arquivo_teste)
)

orcid_teste = extrair_orcid_xml(caminho_teste)

print('arquivo:', arquivo_teste)
print('nome:', nome_teste)
print('id lattes:', id_lattes_teste)
print('orcid:', orcid_teste)

arquivo: /content/drive/MyDrive/RAGI 2026 2/Extrator de Currículo - LATTES/Currículos dos Egressos - LATTES/Abdul_Pedro_Manuel_Muchingeca_9666992319712718.xml
nome: Abdul Pedro Manuel Muchingeca
id lattes: 9666992319712718
orcid: 


## 9. processar todos os currículos

após a validação do teste, todos os arquivos xml encontrados serão processados.

para cada currículo serão registrados o nome do egresso, o id lattes, o orcid, o arquivo de origem e o status da extração.

essa tabela será utilizada como base para iniciar o enriquecimento pela crossref.

In [ ]:
dados_egressos = []
erros_lattes = []

for arquivo in tqdm(arquivos_xml, desc='extraindo orcid dos currículos lattes'):
    # arquivo já é o caminho completo!
    caminho_xml = arquivo

    nome, id_lattes = extrair_identificacao_arquivo(
        os.path.basename(arquivo)
    )

    try:
        orcid = extrair_orcid_xml(caminho_xml)
        status = 'orcid encontrado' if orcid else 'orcid não encontrado'

        dados_egressos.append({
            'Discente': nome,
            'ID_Lattes': id_lattes,
            'ORCID': orcid,
            'Arquivo_XML_Lattes': arquivo,
            'Status_ORCID': status
        })

    except Exception as erro:
        dados_egressos.append({
            'Discente': nome,
            'ID_Lattes': id_lattes,
            'ORCID': '',
            'Arquivo_XML_Lattes': arquivo,
            'Status_ORCID': 'erro'
        })

        erros_lattes.append({
            'Discente': nome,
            'Arquivo_XML_Lattes': arquivo,
            'Erro': str(erro)
        })

extraindo orcid dos currículos lattes:   0%|          | 0/271 [00:00<?, ?it/s]

## 10. visualizar os orcids extraídos

nesta etapa os resultados da extração são transformados em um dataframe.

a tabela permite verificar quantos currículos foram processados e quantos possuem orcid disponível para a consulta à crossref.

In [ ]:
df_egressos = pd.DataFrame(
    dados_egressos
)

print(
    f'total de currículos processados: '
    f'{len(df_egressos)}'
)

print(
    f'egressos com orcid: '
    f'{(df_egressos["ORCID"] != "").sum()}'
)

print(
    f'egressos sem orcid: '
    f'{(df_egressos["ORCID"] == "").sum()}'
)

display(
    df_egressos.head(10)
)

total de currículos processados: 271
egressos com orcid: 79
egressos sem orcid: 192


,Discente,ID_Lattes,ORCID,Arquivo_XML_Lattes,Status_ORCID
0,Abdul Pedro Manuel Muchingeca,9666992319712718,,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,orcid não encontrado
1,Abner Antônio Ferreira,0596211008110321,,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,orcid não encontrado
2,Adriana Teixeira de Moraes,3100726985926660,,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,orcid não encontrado
3,Adrielly Campos e Almeida,4039893861746107,,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,orcid não encontrado
4,Aldenor da Silva Pimentel,5770288768881926,,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,orcid não encontrado
5,Alessandra Rodrigues Oliveira,9237912840073975,0000-0001-8826-5510,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,orcid encontrado
6,Alessandra Siqueira Lessa,5616998500202010,,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,orcid não encontrado
7,Aline Dos Santos,7369015309091344,,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,orcid não encontrado
8,Amanda Costa e Silva,5421893766331754,,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,orcid não encontrado
9,Amanda de Sousa Veloso,2847738122151795,,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,orcid não encontrado


## 11. configurar a api da crossref

nesta etapa são definidos os parâmetros utilizados para acessar a api pública da crossref.

são configurados o endereço da api, o tempo máximo de espera para cada requisição, a quantidade máxima de tentativas em caso de falha e o intervalo entre as consultas.

essas configurações permitem controlar o processo de coleta e reduzir problemas causados por falhas temporárias de comunicação.

In [ ]:
CROSSREF_API = (
    'https://api.crossref.org/works'
)

TIMEOUT = 30

MAX_TENTATIVAS = 3

INTERVALO_ENTRE_EGRESSOS = 0.2


# opcional:
# informe um e-mail para identificação da aplicação

EMAIL_CROSSREF = ''


# limite para testes
# use None para processar todos

LIMITE_PROCESSAMENTO = None


print('api crossref:')
print(CROSSREF_API)

print()

print('timeout:')
print(TIMEOUT)

print()

print('máximo de tentativas:')
print(MAX_TENTATIVAS)

print()

print('limite de processamento:')
print(LIMITE_PROCESSAMENTO)

api crossref:
https://api.crossref.org/works

timeout:
30

máximo de tentativas:
3

limite de processamento:
None


## 12. criar sessão de comunicação

nesta etapa é criada uma sessão para realizar as consultas à api da crossref.

a sessão permite centralizar os cabeçalhos utilizados nas requisições e manter uma configuração única para todo o processamento.

In [ ]:
session = requests.Session()

HEADERS_CROSSREF = {
    'User-Agent': (
        'RAGI-Crossref-Notebook/1.0'
    )
}

if EMAIL_CROSSREF.strip():

    HEADERS_CROSSREF['User-Agent'] += (
        f' mailto:{EMAIL_CROSSREF.strip()}'
    )

print(
    HEADERS_CROSSREF
)

{'User-Agent': 'RAGI-Crossref-Notebook/1.0'}


## 13. testar a consulta à crossref

antes de iniciar o processamento de todos os egressos, será realizada uma consulta utilizando um único orcid encontrado nos currículos lattes.

o objetivo é verificar:

- se o orcid está sendo aceito pela api;
- se a conexão com a crossref está funcionando;
- se existem publicações associadas ao pesquisador;
- quais informações são retornadas pela api.

In [ ]:
def consultar_crossref_por_orcid(orcid):
    if not orcid:
        return [], 'ORCID vazio'

    trabalhos = []
    cursor = '*'

    # Adiciona a URL base do ORCID caso não exista
    orcid_formatado = orcid if orcid.startswith('http') else f'https://orcid.org/{orcid}'

    while True:
        params = {
            'filter': f'orcid:{orcid_formatado}',
            'rows': 100,
            'cursor': cursor
        }

        data = None
        ultimo_erro = ''

        for tentativa in range(1, MAX_TENTATIVAS + 1):
            try:
                response = session.get(
                    CROSSREF_API,
                    params=params,
                    headers=HEADERS_CROSSREF,
                    timeout=TIMEOUT
                )

                if response.status_code == 200:
                    data = response.json()
                    break

                ultimo_erro = f'HTTP {response.status_code}'

            except Exception as erro:
                ultimo_erro = str(erro)

            time.sleep(tentativa)

        if data is None:
            return trabalhos, ultimo_erro

        message = data.get('message', {})
        items = message.get('items', [])
        trabalhos.extend(items)

        next_cursor = message.get('next-cursor')

        if not items or not next_cursor or next_cursor == cursor or len(items) < 100:
            break

        cursor = next_cursor

    return trabalhos, ''

## 14. estruturar os dados das publicações

a api da crossref retorna os metadados das publicações em formato json.

nesta etapa esses dados são transformados em registros tabulares.

cada publicação será relacionada ao respectivo egresso por meio do nome, id lattes e orcid.

também serão armazenados identificadores e metadados da publicação, como doi, título, autores, periódico, issn, editora e ano.

In [ ]:
def extrair_dados_publicacao_crossref(
    pub,
    discente,
    orcid,
    id_lattes
):
    autores = pub.get(
        'author',
        []
    )
    nomes_autores = []

    for autor in autores:
        nome = (
            f"{autor.get('given', '')} "
            f"{autor.get('family', '')}"
        ).strip()

        if nome:
            nomes_autores.append(
                nome
            )
    titulo = pub.get(
        'title',
        []
    )

    if isinstance(titulo, list):
        titulo = (
            titulo[0]
            if titulo
            else ''
        )

    return {
        'Discente': discente,
        'ID_Lattes': id_lattes,
        'ORCID': orcid,
        'Titulo': titulo,
        'Autores': '; '.join(
            nomes_autores
        ),
        'DOI': pub.get(
            'DOI',
            ''
        ),
        'Tipo': pub.get(
            'type',
            ''
        ),
        'Publicado_Ano': (
            pub.get(
                'published-print',
                {}
            )
            .get(
                'date-parts',
                [[]]
            )[0][0]
            if pub.get(
                'published-print',
                {}
            )
            .get(
                'date-parts'
            )
            else ''
        ),
        'Revista': pub.get(
            'container-title',
            ['']
        )[0]
        if pub.get(
            'container-title'
        )
        else '',
        'ISSN': '; '.join(
            pub.get(
                'ISSN',
                []
            )
        ),
        'Editora': pub.get(
            'publisher',
            ''
        ),
        'URL': pub.get(
            'URL',
            ''
        ),
        'Referencias': pub.get(
            'reference-count',
            0
        ),
        'Citado_por': pub.get(
            'is-referenced-by-count',
            0
        )
    }

## 15. processar os egressos na crossref

nesta etapa todos os egressos que possuem um orcid válido serão consultados na api da crossref.

para cada egresso:

1. o orcid é enviado para a api;
2. as publicações retornadas são coletadas;
3. cada publicação é transformada em um registro estruturado;
4. os resultados são associados ao egresso;
5. eventuais erros são registrados separadamente.

os egressos sem orcid não serão consultados, mas continuarão presentes no resumo final.

In [ ]:
todas_publicacoes = []
resumo_egressos = []
erros_processamento = []

df_processamento = df_egressos[
    df_egressos['ORCID'] != ''
].copy()

if LIMITE_PROCESSAMENTO is not None:

    df_processamento = (
        df_processamento
        .head(LIMITE_PROCESSAMENTO)
    )

for _, linha in tqdm(
    df_processamento.iterrows(),
    total=len(df_processamento),
    desc='consultando crossref'
):
    discente = linha[
        'Discente'
    ]
    id_lattes = linha[
        'ID_Lattes'
    ]
    orcid = linha[
        'ORCID'
    ]

    pubs, erro = (
        consultar_crossref_por_orcid(
            orcid
        )
    )

    for pub in pubs:
        todas_publicacoes.append(
            extrair_dados_publicacao_crossref(
                pub,
                discente,
                orcid,
                id_lattes
            )
        )

    resumo_egressos.append({
        'Discente': discente,
        'ID_Lattes': id_lattes,
        'ORCID': orcid,
        'Total_Publicacoes_Crossref': len(
            pubs
        ),
        'Total_Citacoes_Crossref': sum(
            int(
                pub.get(
                    'is-referenced-by-count',
                    0
                ) or 0
            )
            for pub in pubs
        ),
        'Status_Crossref': (
            'concluído'
            if not erro
            else 'erro na consulta'
        ),
        'Erro_Crossref': erro,
        'Data_Consulta_Crossref':
            datetime.now().strftime(
                '%Y-%m-%d %H:%M:%S'
            )
    })

    if erro:
        erros_processamento.append({
            'Discente': discente,
            'ID_Lattes': id_lattes,
            'ORCID': orcid,
            'Erro': erro
        })

    time.sleep(
        INTERVALO_ENTRE_EGRESSOS
    )

consultando crossref:   0%|          | 0/79 [00:00<?, ?it/s]

## 16. incluir egressos sem orcid no resumo

os egressos que não possuem orcid não são enviados para a crossref.

entretanto, eles permanecem no resultado consolidado para que seja possível identificar quais currículos ainda precisam ser enriquecidos ou revisados.

esses registros receberão o status `orcid não encontrado`.

In [ ]:
orcid_processados = {
    item['ORCID']
    for item in resumo_egressos
}

for _, linha in df_egressos.iterrows():
    orcid = linha[
        'ORCID'
    ]

    if orcid:
        continue


    resumo_egressos.append({
        'Discente': linha[
            'Discente'
        ],
        'ID_Lattes': linha[
            'ID_Lattes'
        ],
        'ORCID': '',
        'Total_Publicacoes_Crossref': 0,
        'Total_Citacoes_Crossref': 0,
        'Status_Crossref':
            'ORCID não encontrado',
        'Erro_Crossref':
            'não foi possível consultar a Crossref sem ORCID',
        'Data_Consulta_Crossref':
            datetime.now().strftime(
                '%Y-%m-%d %H:%M:%S'
            )
    })

## 17. criar as tabelas de resultados

nesta etapa os resultados coletados são convertidos em três dataframes.

o primeiro apresenta os indicadores consolidados por egresso.

o segundo apresenta as publicações detalhadas encontradas na crossref.

o terceiro apresenta os erros ocorridos durante o processamento.

In [ ]:
df_pubs = pd.DataFrame(
    todas_publicacoes
)

df_resumo = pd.DataFrame(
    resumo_egressos
)

df_erros = pd.DataFrame(
    erros_processamento
)

print(
    'egressos processados:',
    len(df_resumo)
)

print(
    'publicações encontradas:',
    len(df_pubs)
)

print(
    'egressos com erro:',
    len(df_erros)
)

if not df_resumo.empty:
    display(
        df_resumo.head()
    )

if not df_pubs.empty:
    display(
        df_pubs.head()
    )

egressos processados: 271
publicações encontradas: 220
egressos com erro: 0


,Discente,ID_Lattes,ORCID,Total_Publicacoes_Crossref,Total_Citacoes_Crossref,Status_Crossref,Erro_Crossref,Data_Consulta_Crossref
0,Alessandra Rodrigues Oliveira,9237912840073975,0000-0001-8826-5510,0,0,concluído,,2026-08-27 15:42:40
1,Ana Júlia de Freitas Carrijo,0948153683101920,0000-0001-8372-1003,8,3,concluído,,2026-08-27 15:42:41
2,Andréa Pereira dos Santos,9315618025567235,0000-0001-5410-5500,17,8,concluído,,2026-08-27 15:42:42
3,Antônio Guilherme de Lima Santos,9779435301297884,0000-0002-4218-5627,0,0,concluído,,2026-08-27 15:42:43
4,Augusto Flamaryon Cecchin Bozz,9837974802580178,0000-0002-5723-2158,3,1,concluído,,2026-08-27 15:42:44


,Discente,ID_Lattes,ORCID,Titulo,Autores,DOI,Tipo,Publicado_Ano,Revista,ISSN,Editora,URL,Referencias,Citado_por
0,Ana Júlia de Freitas Carrijo,0948153683101920,0000-0001-8372-1003,A identificação do público infantil com &lt;i&...,Lara Lima Satler; Ana Júlia de Freitas Carrijo,10.20435/multi.v24i56.2144,journal-article,,Multitemas,2447-9276; 1414-512X,Universidade Catolica Dom Bosco,https://doi.org/10.20435/multi.v24i56.2144,0,0
1,Ana Júlia de Freitas Carrijo,0948153683101920,0000-0001-8372-1003,Articulações teóricometodológicas em uma exper...,Lara Lima Satler; Ana Júlia de Freitas Carrijo,10.19132/1807-8583202253.111888,journal-article,,Intexto,1807-8583,Universidade Federal do Rio Grande do Sul,https://doi.org/10.19132/1807-8583202253.111888,25,0
2,Ana Júlia de Freitas Carrijo,0948153683101920,0000-0001-8372-1003,Padrões de performance no Instagram na experi...,Ana Júlia de Freitas Carrijo; LARA LIMA SATLER,10.29146/ecopos.v24i3.27666,journal-article,,Revista ECO-Pós,2175-8689; 2175-8689,Revista ECO-Pos,https://doi.org/10.29146/ecopos.v24i3.27666,0,0
3,Ana Júlia de Freitas Carrijo,0948153683101920,0000-0001-8372-1003,Algoritmos e cultura digital,Ana Júlia De Freitas Carrijo; Ana Carolina Dam...,10.15448/1980-3729.2024.1.44941,journal-article,,Revista FAMECOS,1980-3729; 1415-0549,EDIPUCRS,https://doi.org/10.15448/1980-3729.2024.1.44941,52,2
4,Ana Júlia de Freitas Carrijo,0948153683101920,0000-0001-8372-1003,Rumo a uma abordagem sociocultural de tecnolog...,Ana Júlia de Freitas Carrijo; Ana Carolina Dam...,10.34019/1981-4070.2023.v17.41094,journal-article,,Lumina,1981-4070; 1516-0785,Universidade Federal de Juiz de Fora,https://doi.org/10.34019/1981-4070.2023.v17.41094,0,1


In [ ]:
with pd.ExcelWriter(
    ARQUIVO_SAIDA,
    engine='openpyxl'
) as writer:

    df_resumo.to_excel(
        writer,
        sheet_name='Resumo de Indicadores',
        index=False
    )

    df_pubs.to_excel(
        writer,
        sheet_name='Publicações Detalhadas',
        index=False
    )

    df_erros.to_excel(
        writer,
        sheet_name='Erros Crossref',
        index=False
    )

print(
    'arquivo salvo com sucesso:'
)

print(
    ARQUIVO_SAIDA
)

arquivo salvo com sucesso:
/content/drive/MyDrive/RAGI 2026 2/Enriquecimento de dados - CROSSREF/Estratégia 1/publicacoes_crossref_lattes.xlsx


## 18. resumo da execução

ao final do processamento são apresentados os principais resultados da execução.

essa etapa permite verificar rapidamente a quantidade de currículos processados, egressos com orcid, publicações encontradas e erros registrados.

In [ ]:
print(
    '=============================================='
)
print(
    'resumo da execução crossref'
)
print(
    '=============================================='
)
print(
    f'currículos xml encontrados: '
    f'{len(arquivos_xml)}'
)
print(
    f'egressos processados: '
    f'{len(df_resumo)}'
)
print(
    f'egressos com orcid: '
    f'{(df_egressos["ORCID"] != "").sum()}'
)
print(
    f'egressos sem orcid: '
    f'{(df_egressos["ORCID"] == "").sum()}'
)
print(
    f'publicações encontradas: '
    f'{len(df_pubs)}'
)
print(
    f'consultas com erro: '
    f'{len(df_erros)}'
)
print(
    f'arquivo de saída: '
    f'{ARQUIVO_SAIDA}'
)

resumo da execução crossref
currículos xml encontrados: 271
egressos processados: 271
egressos com orcid: 79
egressos sem orcid: 192
publicações encontradas: 220
consultas com erro: 0
arquivo de saída: /content/drive/MyDrive/RAGI 2026 2/Enriquecimento de dados - CROSSREF/Estratégia 1/publicacoes_crossref_lattes.xlsx
